# Fase 1 · M04a: Unión de Tablas

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 1 — Transformación |
| **Módulo** | M04a — Unión de Tablas |

---

## 🎯 Qué hace

Une los parquets por tabla (interim) en un único dataframe de alumnos a nivel de expediente.

## 📋 Requisitos

- `data/01_interim/*.parquet` — tablas individuales por dominio

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/02_processed/df_alumno.parquet` | Dataset integrado de alumnos |

## 🔄 Flujo

```
data/01_interim/*.parquet
    ↓ merge por per_id_ficticio
    → data/02_processed/df_alumno.parquet
```

## ➡️ Siguiente

`f1_m04c_correccion_notas.ipynb` — corrección de notas de acceso


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
#
# Importa librerías y configuración del proyecto.
#
# De src.constantes importamos:
#   - TABLA_PREINSCRIPCION: nombre de la tabla de preinscripción ('preinscripcion')
#     Se excluye aquí porque se une en M04b.
#   - TABLAS_EXCEL_PRINCIPAL: lista de las 8 tablas del Excel principal
#     ['expedientes', 'titulaciones', 'demograficos', ...]
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# --- Detectar entorno ---
# En Colab: monta Drive. En Local: busca src/ subiendo niveles.
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

if not (ROOT / 'src').exists():
    raise FileNotFoundError(
        f'No se encontró la carpeta src/ en {ROOT}\n'
        f'Asegúrate de que este notebook está dentro del proyecto (AU_UJI/)'
    )

sys.path.insert(0, str(ROOT))

# --- Imports del proyecto ---
from src.config import RUTA_INTERIM, RUTA_PROCESSED, info_entorno
from src.constantes import TABLA_PREINSCRIPCION, TABLAS_EXCEL_PRINCIPAL
from src.utils import crear_directorios, formato_numero_es

# Crear carpeta de salida
crear_directorios([RUTA_PROCESSED])

# Atajo para formato español
fmt = formato_numero_es

# Mostrar entorno
info_entorno()

print(f'\n📋 Tablas a unir: {len(TABLAS_EXCEL_PRINCIPAL)}')
print(f'   Excluida: {TABLA_PREINSCRIPCION} (se une en M04b)')

✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR TABLAS LIMPIAS (desde parquets)
# ============================================================================
#
# Lee dinámicamente todos los parquets de data/01_interim/,
# EXCEPTO preinscripcion (que se une en M04b).
#
# La lectura es dinámica: no asumimos qué archivos hay,
# sino que buscamos todos los .parquet y excluimos preinscripción.
# ============================================================================

print('=' * 60)
print('CARGANDO TABLAS LIMPIAS')
print('=' * 60)

# Diccionario: nombre_tabla → DataFrame
tablas = {}

for ruta in sorted(RUTA_INTERIM.glob('*.parquet')):
    # Excluir preinscripción (se une en M04b)
    if ruta.stem != TABLA_PREINSCRIPCION:
        tablas[ruta.stem] = pd.read_parquet(ruta)
        print(f'   ✅ {ruta.stem:15s}: {fmt(len(tablas[ruta.stem])):>10s} registros')

print(f'\n✅ Tablas cargadas: {len(tablas)}')

CARGANDO TABLAS LIMPIAS
   ✅ becas          :     70.524 registros
   ✅ demograficos   :     30.873 registros
   ✅ domicilios     :    210.911 registros
   ✅ expedientes    :    109.568 registros
   ✅ notas          :    107.908 registros


   ✅ recibos        :    114.454 registros
   ✅ titulaciones   :         45 registros
   ✅ trabajo        :    195.524 registros

✅ Tablas cargadas: 8


In [3]:
# ============================================================================
# CELDA 3: PROCESAR DOMICILIOS → variable 'vive_fuera'
# ============================================================================
#
# La tabla domicilios tiene 2 tipos de dirección por alumno y curso:
#   - 'Familiar/Personal': donde vive su familia
#   - 'Durante el curso': donde vive mientras estudia
#
# Si estas 2 poblaciones son diferentes → el alumno 'vive_fuera' = True.
# Esta variable es importante para predecir abandono: los alumnos que
# se desplazan para estudiar tienen patrones diferentes.
#
# Proceso:
#   1. Separar domicilios por tipo (Familiar vs Curso)
#   2. Eliminar duplicados (un alumno puede tener varios registros)
#   3. Unir ambos tipos por (per_id_ficticio, curso_aca)
#   4. Comparar poblaciones → crear 'vive_fuera'
#   5. Consolidar columnas de ubicación (poblacion, provincia, pais)
# ============================================================================

print('=' * 60)
print('PROCESANDO DOMICILIOS')
print('=' * 60)

df_dom_final = None

if 'domicilios' in tablas:
    df_dom = tablas['domicilios'].copy()

    # --- Paso 1: Separar por tipo de domicilio ---
    df_familiar = df_dom[df_dom['tipo_domicilio'] == 'Familiar/Personal'].copy()
    df_curso = df_dom[df_dom['tipo_domicilio'] == 'Durante el curso'].copy()

    # --- Paso 2: Eliminar duplicados ---
    # Un alumno puede tener varios domicilios del mismo tipo en un curso.
    # Nos quedamos con el primero (keep='first').
    df_familiar = df_familiar.drop_duplicates(
        subset=['per_id_ficticio', 'curso_aca'], keep='first'
    )
    df_curso = df_curso.drop_duplicates(
        subset=['per_id_ficticio', 'curso_aca'], keep='first'
    )

    # --- Paso 3: Renombrar columnas para distinguir origen ---
    df_familiar = df_familiar.rename(columns={
        'poblacion': 'pob_familiar',
        'provincia': 'prov_familiar',
        'pais': 'pais_familiar'
    })
    df_familiar = df_familiar[['per_id_ficticio', 'curso_aca',
                               'pob_familiar', 'prov_familiar', 'pais_familiar']]

    df_curso = df_curso.rename(columns={
        'poblacion': 'pob_curso',
        'provincia': 'prov_curso',
        'pais': 'pais_curso'
    })
    df_curso = df_curso[['per_id_ficticio', 'curso_aca',
                         'pob_curso', 'prov_curso', 'pais_curso']]

    # --- Paso 4: Unir familiar + curso ---
    # outer join: mantenemos alumnos que solo tienen un tipo de domicilio
    df_dom_proc = df_familiar.merge(
        df_curso, on=['per_id_ficticio', 'curso_aca'], how='outer'
    )

    # --- Paso 5: Crear 'vive_fuera' ---
    # True si la población familiar ≠ población durante el curso
    # (y ambas están informadas)
    df_dom_proc['vive_fuera'] = (
        (df_dom_proc['pob_familiar'] != df_dom_proc['pob_curso']) &
        df_dom_proc['pob_curso'].notna() &
        df_dom_proc['pob_familiar'].notna()
    )

    # --- Paso 6: Consolidar columnas ---
    # Priorizar dirección de curso; si no existe, usar la familiar
    df_dom_proc['poblacion'] = df_dom_proc['pob_curso'].fillna(df_dom_proc['pob_familiar'])
    df_dom_proc['provincia'] = df_dom_proc['prov_curso'].fillna(df_dom_proc['prov_familiar'])
    df_dom_proc['pais_domicilio'] = df_dom_proc['pais_curso'].fillna(df_dom_proc['pais_familiar'])

    # Columnas finales (solo las que necesitamos para el merge)
    df_dom_final = df_dom_proc[[
        'per_id_ficticio', 'curso_aca',
        'poblacion', 'provincia', 'pais_domicilio', 'vive_fuera'
    ]]

    pct_vive_fuera = df_dom_final['vive_fuera'].mean() * 100
    print(f"\n✅ vive_fuera: {fmt(df_dom_final['vive_fuera'].sum())} True ({pct_vive_fuera:.1f}%)")
else:
    print('⚠️ Tabla domicilios no encontrada')

PROCESANDO DOMICILIOS



✅ vive_fuera: 13.990 True (12.8%)


In [4]:
# ============================================================================
# CELDA 4: UNIR LAS 8 TABLAS
# ============================================================================
#
# La tabla central es EXPEDIENTES (un registro por alumno×titulación×curso).
# Sobre ella hacemos left join de las demás tablas.
#
# ¿Por qué left join?
#   Queremos mantener TODOS los registros de expedientes (la base),
#   y añadir información complementaria cuando exista.
#   Si un alumno no tiene beca, la columna 'nombre_beca' queda NaN.
#
# Claves de merge:
#   - titulaciones:  exp_tit_id (código de titulación)
#   - demograficos:  per_id_ficticio (código de alumno)
#   - domicilios:    per_id_ficticio + curso_aca
#   - becas:         per_id_ficticio + curso_aca
#   - trabajo:       per_id_ficticio + exp_tit_id + curso_aca
#   - notas:         per_id_ficticio + exp_tit_id + curso_aca
#   - recibos:       per_id_ficticio + curso_aca
#
# IMPORTANTE: verificamos que len(df) no cambia tras cada merge.
# Si cambia, hay duplicados en la tabla que estamos uniendo.
# ============================================================================

print('=' * 60)
print('UNIENDO TABLAS')
print('=' * 60)

# --- 1. Base: Expedientes ---
if 'expedientes' not in tablas:
    raise ValueError('Tabla expedientes no encontrada — es la base del merge')

df = tablas['expedientes'].copy()
n_base = len(df)
print(f'1. Expedientes (base): {fmt(len(df))}')

# --- 2. + Titulaciones (por exp_tit_id) ---
if 'titulaciones' in tablas:
    cols_tit = [c for c in ['exp_tit_id', 'titulacion', 'rama', 'cred_titulacion']
                if c in tablas['titulaciones'].columns]
    df = df.merge(tablas['titulaciones'][cols_tit], on='exp_tit_id', how='left')
    assert len(df) == n_base, f'¡Duplicados tras merge titulaciones! {len(df)} != {n_base}'
    print(f'2. + Titulaciones: {fmt(len(df))}')

# --- 3. + Demográficos (por per_id_ficticio) ---
if 'demograficos' in tablas:
    df = df.merge(tablas['demograficos'], on='per_id_ficticio', how='left')
    assert len(df) == n_base, f'¡Duplicados tras merge demograficos! {len(df)} != {n_base}'
    print(f'3. + Demograficos: {fmt(len(df))}')

# --- 4. + Domicilios (por per_id + curso) ---
if df_dom_final is not None:
    df = df.merge(df_dom_final, on=['per_id_ficticio', 'curso_aca'], how='left')
    assert len(df) == n_base, f'¡Duplicados tras merge domicilios! {len(df)} != {n_base}'
    print(f'4. + Domicilios: {fmt(len(df))}')

# --- 5. + Becas (por per_id + curso) ---
if 'becas' in tablas:
    df_becas = tablas['becas'].copy()
    # Renombrar columna de curso si es necesario
    if 'mat_curso_aca' in df_becas.columns:
        df_becas = df_becas.rename(columns={'mat_curso_aca': 'curso_aca'})
    # Eliminar duplicados (un alumno puede tener varias becas en un curso)
    df_becas = df_becas.drop_duplicates(
        subset=['per_id_ficticio', 'curso_aca'], keep='first'
    )
    cols_beca = ['per_id_ficticio', 'curso_aca'] + \
                [c for c in ['nombre_beca'] if c in df_becas.columns]
    df = df.merge(df_becas[cols_beca], on=['per_id_ficticio', 'curso_aca'], how='left')
    # Crear variable derivada: tiene_beca (True/False)
    df['tiene_beca'] = df['nombre_beca'].notna() if 'nombre_beca' in df.columns else False
    assert len(df) == n_base, f'¡Duplicados tras merge becas! {len(df)} != {n_base}'
    print(f'5. + Becas: {fmt(len(df))}')

# --- 6. + Trabajo (por per_id + tit + curso) ---
if 'trabajo' in tablas:
    df_trabajo = tablas['trabajo'].copy()
    if 'mat_curso_aca' in df_trabajo.columns:
        df_trabajo = df_trabajo.rename(columns={'mat_curso_aca': 'curso_aca'})
    df_trabajo = df_trabajo.drop_duplicates(
        subset=['per_id_ficticio', 'exp_tit_id', 'curso_aca'], keep='first'
    )
    cols_trab = ['per_id_ficticio', 'exp_tit_id', 'curso_aca'] + \
                [c for c in ['nombre_trabajo'] if c in df_trabajo.columns]
    df = df.merge(df_trabajo[cols_trab],
                  on=['per_id_ficticio', 'exp_tit_id', 'curso_aca'], how='left')
    assert len(df) == n_base, f'¡Duplicados tras merge trabajo! {len(df)} != {n_base}'
    print(f'6. + Trabajo: {fmt(len(df))}')

# --- 7. + Notas (por per_id + tit + curso) ---
if 'notas' in tablas:
    cols_notas = ['per_id_ficticio', 'exp_tit_id', 'curso_aca'] + \
                 [c for c in ['media_titulacion_curso', 'media_titulacion_alumno']
                  if c in tablas['notas'].columns]
    df = df.merge(tablas['notas'][cols_notas],
                  on=['per_id_ficticio', 'exp_tit_id', 'curso_aca'], how='left')
    assert len(df) == n_base, f'¡Duplicados tras merge notas! {len(df)} != {n_base}'
    print(f'7. + Notas: {fmt(len(df))}')

# --- 8. + Recibos (por per_id + curso) ---
if 'recibos' in tablas:
    df_recibos = tablas['recibos'].drop_duplicates(
        subset=['per_id_ficticio', 'curso_aca'], keep='first'
    )
    cols_rec = ['per_id_ficticio', 'curso_aca'] + \
               [c for c in ['forma_de_pago', 'numero_pagos'] if c in df_recibos.columns]
    df = df.merge(df_recibos[cols_rec],
                  on=['per_id_ficticio', 'curso_aca'], how='left')
    assert len(df) == n_base, f'¡Duplicados tras merge recibos! {len(df)} != {n_base}'
    print(f'8. + Recibos: {fmt(len(df))}')

# Renombrar para claridad
df_alumno = df

print(f'\n✅ Dataset base: {fmt(len(df_alumno))} registros × {len(df_alumno.columns)} columnas')

UNIENDO TABLAS
1. Expedientes (base): 109.568
2. + Titulaciones: 109.568
3. + Demograficos: 109.568
4. + Domicilios: 109.568


5. + Becas: 109.568
6. + Trabajo: 109.568
7. + Notas: 109.568


8. + Recibos: 109.568

✅ Dataset base: 109.568 registros × 33 columnas


In [5]:
# ============================================================================
# CELDA 5: GUARDAR DATASET BASE
# ============================================================================
#
# Guarda el resultado como df_alumno_base.parquet en data/02_processed/.
# Este es un dataset INTERMEDIO — el definitivo (df_alumno.parquet)
# se genera en M04b tras añadir preinscripción.
#
# ¿Por qué guardar un intermedio?
#   - Permite verificar el merge antes de añadir preinscripción
#   - Si M04b falla, no hay que repetir M04a
#   - Facilita la depuración paso a paso
# ============================================================================

print('=' * 60)
print('GUARDANDO')
print('=' * 60)

# Guardar parquet
path = RUTA_PROCESSED / 'df_alumno_base.parquet'
df_alumno.to_parquet(path, index=False)

# Mostrar tamaño del archivo
size_mb = path.stat().st_size / 1024 / 1024
print(f'\n✅ Guardado: {path.name} ({size_mb:.2f} MB)')
print(f'   Ruta: {path}')
print(f'   Dimensiones: {fmt(len(df_alumno))} registros × {len(df_alumno.columns)} columnas')
print(f'\n📌 Siguiente: f1_m04b_union_preinscripcion.ipynb')

GUARDANDO



✅ Guardado: df_alumno_base.parquet (2.18 MB)
   Ruta: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed\df_alumno_base.parquet
   Dimensiones: 109.568 registros × 33 columnas

📌 Siguiente: f1_m04b_union_preinscripcion.ipynb
